In [290]:
import os
import numpy as np
import mne
from pathlib import Path

In [291]:
subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
subjects_dir = Path(subjects_dir).parent
subjects_dir
subject = 'fsaverage'

0 files missing from root.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data
0 files missing from bem.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage


In [292]:
src = mne.setup_source_space(subject = subject, spacing='ico5', subjects_dir=subjects_dir, add_dist=False)

Setting up the source space with the following parameters:

SUBJECTS_DIR = C:\Users\hugop\mne_data\MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 5

>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.white...
Mapping lh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 10242/163842 selected to source space (ico = 5)

Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.white...
Mapping rh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.sphere...
Setting up the triangulation for the decimated surface...
loaded rh.white 10242/163842 selected to 

In [293]:
hemi = 0 # 0 = left | 1 = right

rr = src[hemi]['rr']
vertno = src[hemi]['vertno'] 
coords = rr[vertno] 

In [294]:
y_ant = 0.02    # avant
y_post = -0.04  # arrière
z_sup = 0.02    # haut
z_inf = -0.02   # bas

x = coords[:, 0]
y = coords[:, 1]
z = coords[:, 2]

In [295]:
print(coords.min(axis=0))
print(coords.max(axis=0))

xmin, ymin, zmin = coords.min(axis=0)
xmax, ymax, zmax = coords.max(axis=0)

temporal_mask = (z < z_inf) & (y > y_post) & (y < y_ant)
temporal_vertices = vertno[temporal_mask]

occipital_mask = y < y_post
occipital_vertices = vertno[occipital_mask]

frontal_mask = y > y_ant
frontal_vertices = vertno[frontal_mask]

used = temporal_mask | occipital_mask | frontal_mask
parietal_mask = ~used
parietal_vertices = vertno[parietal_mask]

[-0.0656627  -0.10240875 -0.04242961]
[0.00179359 0.0647402  0.07524361]


In [296]:
temp_coords = coords[temporal_mask]
temp_vert = vertno[temporal_mask]

temp_ant = temp_vert[temp_coords[:, 1] > 0]
temp_post = temp_vert[temp_coords[:, 1] <= 0]

temp_sup = temp_vert[temp_coords[:, 2] > -0.04]
temp_inf = temp_vert[temp_coords[:, 2] <= -0.04]


In [297]:
atlas = {
    "frontal": frontal_vertices,
    "parietal": parietal_vertices,
    "occipital": occipital_vertices,
    "temporal": temporal_vertices,
    
    "temporal_ant": temp_ant,
    "temporal_post": temp_post,
    "temporal_sup": temp_sup,
    "temporal_inf": temp_inf,
}

In [298]:

label = mne.Label(
    vertices=temporal_vertices,
    hemi='lh',
    name='temporal_custom'
)


In [299]:
def make_mask(coords, px = (0,1),py= (0,1),pz= (0,1)):
    
    def calcul(min,max,p):
        return min + p * (max - min)
    
    minx,miny,minz = coords.min(axis=0)
    maxx, maxy, maxz = coords.max(axis=0)

    x = coords[:, 0]
    y = coords[:, 1]
    z = coords[:, 2]

    x_inf = calcul(minx,maxx,px[0])
    x_sup = calcul(minx,maxx,px[1])

    y_inf = calcul(miny,maxy,py[0])
    y_sup = calcul(miny,maxy,py[1])

    z_inf = calcul(minz,maxz,pz[0])
    z_sup = calcul(minz,maxz,pz[1])


    mask = ((x <= x_sup) & (x >= x_inf) & 
             (y <= y_sup) & (y >= y_inf) & 
             (z <= z_sup) & (z >= z_inf))

    return mask

In [300]:
brain = mne.viz.Brain('fsaverage', hemi='lh', surf='inflated')

mask_test2 = make_mask(coords,py=(0.35,0.47),pz=(0,0.5),px = (0,0.37))
mask_test_bis = make_mask(coords,py=(0.47,0.6),pz=(0,0.42),px = (0,0.36))
mask_test = make_mask(coords,py=(0.47,0.75),pz=(0,0.366),px = (0,0.36))
mask_test3 = make_mask(coords,py=(0.22,0.35),pz=(0,0.55),px = (0,0.45))


test = mne.Label(vertno[mask_test], hemi='lh', name='test')
test_bis = mne.Label(vertno[mask_test_bis], hemi='lh', name='test')
test2 = mne.Label(vertno[mask_test2], hemi='lh', name='test')
test3 = mne.Label(vertno[mask_test3], hemi='lh', name='test')



# mask = coords[:,1] 
# verts = vertno[mask]
# test = mne.Label(verts, hemi='lh', name='test')
# all = mne.Label(vertno, hemi='rh', name='test')

brain.add_label(test, color='yellow')
brain.add_label(test_bis, color='yellow')
brain.add_label(test2, color='red')
brain.add_label(test3, color='blue')




# brain.add_label(all, color='blue')


mask.shape
# coords.shape



(10242,)